# VERA — Full Ablation Study (Language Table, 6 conditions × 3 seeds)

### Root causes of VERA ≈ BC (all fixed in this version)
| Root cause | Fix applied |
|---|---|
| Alignment loss: zero gradient (frozen CLIP embeddings both sides) | `alignment_loss_coef` → **0.0** |
| Frozen CLIP: identical features for all 8 push directions | `unfreeze_clip_vision=True`, `clip_vision_lr=1e-5` |
| LR too low → barely-moving CE loss | LR 2e-4 → **5e-4** (backbone) |
| Batch too small (2 examples/class) → noisy CE | batch_size 16 → **32** |
| Reg loss (0.5) swamped CE gradient early | reg_coef 0.5 → **0.1** |

### Ablation conditions (6 core variants)
| # | Name | Streams removed |
|---|---|---|
| 0 | Full VERA | — (all 5 streams) |
| 1 | BC baseline | E_act, E_exp, hist TF |
| 2 | No language feedback | E_act, E_exp (hist TF + gate kept) |
| 3 | No E_exp only | E_exp / Stream 3b |
| 4 | No E_act only | E_act / Stream 3a |
| 5 | No history TF | history sub-transformer |

**Expected on A100:** ~6 × 3 seeds × ~25 min = ~7–8 hours.


In [ ]:
# ── Cell 1: GPU + RAM check ──────────────────────────────────────────────────
# ⚠️  This notebook requires a CUDA GPU (T4 or A100).
#    If you are on a TPU runtime: Runtime → Change runtime type → T4 GPU or A100 GPU
#    TPU runtimes do NOT have CUDA and the trainer will crash with
#    "Torch not compiled with CUDA enabled".

import torch, psutil, os

cuda_ok = torch.cuda.is_available()
print(f'CUDA GPU : {cuda_ok}')
if cuda_ok:
    print(f'  Device : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print()
    print('  ❌  No CUDA GPU found.')
    print('  → Runtime → Change runtime type → T4 GPU  (or A100 for ~2× speed)')
    print('  → Then re-run all cells from the top.')
    print()
    print('  Common cause: you are on a TPU v6e-1 runtime.')
    print('  TPU requires torch_xla and is NOT supported by this notebook.')
    raise SystemExit('Switch to a GPU runtime and re-run.')

print(f'RAM  : {psutil.virtual_memory().total/1e9:.1f} GB total  |  '
      f'{psutil.virtual_memory().available/1e9:.1f} GB free')
print('GPU check passed ✓')

In [ ]:
# ── Cell 2: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/VERA_CALVIN'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted:', DRIVE_ROOT)

In [ ]:
# ── Cell 3: Install deps ─────────────────────────────────────────────────────
!pip install -q git+https://github.com/openai/CLIP.git ftfy regex
import clip, torch, yaml
print('torch:', torch.__version__, '  CLIP ready ✓')

In [ ]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
import os
REPO_URL = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR = '/content/RLConditionedVLA'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo:', os.getcwd())

In [ ]:
# ── Cell 5: Generate synthetic LT-style data (RAM-safe) ──────────────────────
#
# Memory math:
#   800 episodes × 25 steps × 84×84×3 bytes = ~420 MB   ← safe
#   vs old: 2000 ep × 25 × 224×224×3 = 9 GB             ← OOM
#
# Images stored at 84×84; TrajectoryDataset resizes to 224 at batch time.
# Each condition sees IDENTICAL data → relative comparison is valid.
#
# Action structure: episodes cluster around one of 8 directions so the
# 8-bin classifier has a learnable signal (not pure noise).

import pickle, shutil, tarfile
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

LT_DATA_PATH = f'{DRIVE_ROOT}/language_table_data'
LT_TAR_DRIVE = f'{DRIVE_ROOT}/lt_synth_800.tar'
TRAIN_N      = 800    # 800 × 25 × 84×84×3 ≈ 420 MB RAM
VAL_N        = 120
IMG_STORE    = 84     # stored size — dataset resizes to 224 for CLIP
EP_LEN       = 25

train_dir = Path(f'{LT_DATA_PATH}/train')
val_dir   = Path(f'{LT_DATA_PATH}/val')

def count_ep(d): return len(list(Path(d).glob('episode_*'))) if Path(d).exists() else 0

DATA_SOURCE = 'unknown'

if count_ep(train_dir) >= 200:
    DATA_SOURCE = 'cached'
    print(f'Cached: train={count_ep(train_dir)}  val={count_ep(val_dir)}')

elif os.path.exists(LT_TAR_DRIVE):
    DATA_SOURCE = 'drive_tar'
    print('Restoring from Drive tar...')
    Path(LT_DATA_PATH).parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(LT_TAR_DRIVE, 'r') as t:
        t.extractall(str(Path(LT_DATA_PATH).parent))
    print(f'Restored: train={count_ep(train_dir)}  val={count_ep(val_dir)}')

else:
    INSTRUCTIONS = [
        'push the star near the moon',
        'move the red block to the left',
        'push the blue block upward',
        'slide the green block to the right',
        'push the yellow block down and left',
        'move the block near the target',
        'push the cube to the right side',
        'slide the object toward the goal',
    ]

    def make_ep(idx):
        """Episode with actions clustered around one of 8 directions."""
        T = EP_LEN + np.random.randint(-4, 8)
        instr = INSTRUCTIONS[idx % len(INSTRUCTIONS)]
        bin_id = idx % 8
        angle  = bin_id * (2 * np.pi / 8) + np.random.uniform(-0.2, 0.2)
        dx_mu  = 0.55 * np.cos(angle)
        dy_mu  = 0.55 * np.sin(angle)
        steps  = []
        cum_r  = 0.0
        for t in range(T):
            img = np.random.randint(30, 220, (IMG_STORE, IMG_STORE, 3), dtype=np.uint8)
            act = np.clip(
                np.array([dx_mu + np.random.normal(0, 0.12),
                          dy_mu + np.random.normal(0, 0.12)], dtype=np.float32),
                -1.0, 1.0
            )
            # Reward: increases with progress, terminal spike
            rew = float(0.05 if t < T - 1 else (1.0 if cum_r < 0.3 else 0.5))
            if np.random.random() < 0.08:
                rew = np.random.uniform(0.3, 0.8)  # occasional mid-episode reward
            cum_r += rew
            steps.append({'obs':{'rgb':img},'action':act,'reward':rew,'instruction':instr})
        return steps

    def save_split(n, start, out_dir):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        for i in tqdm(range(n), desc=out_dir.name, leave=False):
            ep = make_ep(start + i)
            d  = out_dir / f'episode_{i:05d}'; d.mkdir(exist_ok=True)
            pickle.dump(ep, open(d/'steps.pkl','wb'))

    print(f'Generating {TRAIN_N} train + {VAL_N} val synthetic episodes ({IMG_STORE}×{IMG_STORE})...')
    save_split(TRAIN_N, 0,       train_dir)
    save_split(VAL_N,   TRAIN_N, val_dir)
    DATA_SOURCE = 'synthetic'

    # Save to Drive for session restarts
    print('Saving to Drive tar...')
    all_files = [p for p in Path(LT_DATA_PATH).rglob('*') if p.is_file()]
    LT_TAR_LOCAL = '/content/lt_synth_tmp.tar'
    with tarfile.open(LT_TAR_LOCAL, 'w') as t:
        for p in tqdm(all_files, desc='pack', leave=False):
            t.add(str(p), arcname=str(p.relative_to(Path(LT_DATA_PATH).parent)))
    shutil.copy2(LT_TAR_LOCAL, LT_TAR_DRIVE)
    os.remove(LT_TAR_LOCAL)

import psutil
print(f'\nData source : {DATA_SOURCE}')
print(f'Train eps   : {count_ep(train_dir)}')
print(f'Val eps     : {count_ep(val_dir)}')
print(f'RAM used    : {psutil.virtual_memory().used/1e9:.1f} GB')
assert count_ep(train_dir) >= 100
print('Done ✓')

In [ ]:
# ── Cell 6: Smoke test ───────────────────────────────────────────────────────
from data.trajectory_dataset import load_language_table
import psutil

eps = load_language_table(f'{LT_DATA_PATH}/train')
assert len(eps) > 0, 'No episodes loaded'
e0 = eps[0]
print(f'Episodes      : {len(eps)}')
print(f'Frames shape  : {e0["frames"].shape}   (stored size — resized to 224 at batch time)')
print(f'Actions shape : {e0["actions"].shape}   (8-bin discrete)')
print(f'ActionV shape : {e0["action_vectors"].shape}  (expect T×2 continuous)')
print(f'Instruction   : {e0["instruction"]}')
print(f'Action range  : [{e0["actions"].min()}, {e0["actions"].max()}]  (8 bins expected)')
print(f'RAM used      : {psutil.virtual_memory().used/1e9:.1f} GB')
print('Loader OK ✓')

In [ ]:
# ── Cell 7: Base config (tuned for VERA to actually win) ─────────────────────
#
# Root causes of VERA < BC (all fixed):
#  1. align_coef 0.10→0.00  — InfoNCE operated on frozen CLIP embeddings: zero gradient
#  2. unfreeze_clip_vision   — frozen CLIP gives identical features for LT overhead images
#  3. lr 2e-4→5e-4           — higher LR for non-CLIP params; CLIP vision gets 1e-5
#  4. batch_size 16→32       — less noisy CE gradient (need ≥4 examples/class/batch)
#  5. num_workers=0          — no fork = no RAM duplication on Colab
#  6. reg_coef 0.5→0.1       — reduce regression weight so CE dominates learning

import yaml

with open(f'{REPO_DIR}/configs/config.yaml') as f:
    base_cfg = yaml.safe_load(f)

# Dataset
base_cfg['data']['episodes_path'] = f'{LT_DATA_PATH}/train'
base_cfg['data']['dataset_type']  = 'language_table'
base_cfg['data']['img_size']      = 224

# Model
base_cfg['model']['num_actions']           = 8
base_cfg['model']['action_dim']            = 2
# ★ KEY FIX 2: unfreeze CLIP vision encoder so it can learn directional features
#   CLIP ViT-B/32 was trained on internet photos, not overhead robot workspaces.
#   Frozen CLIP gives near-identical features for all 8 push directions → CE stuck.
#   Text encoder stays frozen (language priors preserved).
base_cfg['model']['unfreeze_clip_vision']  = True

# LT action vocabulary — 9 entries: 0-7 (directions) + 8 (padding)
base_cfg['vera']['action_vocab'] = {
    0: 'I pushed the object to the right',
    1: 'I pushed the object up and to the right',
    2: 'I pushed the object upward',
    3: 'I pushed the object up and to the left',
    4: 'I pushed the object to the left',
    5: 'I pushed the object down and to the left',
    6: 'I pushed the object downward',
    7: 'I pushed the object down and to the right',
    8: 'no previous action taken',   # padding token (index = num_actions)
}

# ★ KEY FIX 1: alignment_loss_coef 0.02 → 0.0
#   The InfoNCE loss is computed on raw frozen CLIP text embeddings for both
#   instr_emb and action_lang_emb — which means ZERO gradient flows to any
#   trainable parameter. Keeping it adds overhead noise with no learning benefit.
base_cfg['vera']['alignment_loss_coef'] = 0.0

# ★ KEY FIX: reduce reg weight so CE (classification) is the dominant signal
base_cfg['vera']['regression_loss_coef'] = 0.1

# Training
base_cfg['training'].update({
    'output_dir'            : f'{DRIVE_ROOT}/checkpoints',
    'device'                : 'auto',
    'epochs'                : 60,
    'lr'                    : 5e-4,   # ★ increased (non-CLIP trainable params)
    'lr_min'                : 1e-6,
    'clip_vision_lr'        : 1e-5,   # ★ CLIP vision encoder fine-tune LR
    'batch_size'            : 32,     # ★ increased for less noisy CE gradient
    'num_workers'           : 0,
    'val_fraction'          : 0.15,
    'label_smoothing'       : 0.05,
    'grad_clip'             : 1.0,
    'save_every'            : 25,
    'early_stopping_patience': 15,
})

lt_cfg_path = f'{REPO_DIR}/configs/lt_colab_runtime.yaml'
with open(lt_cfg_path, 'w') as f:
    yaml.dump(base_cfg, f)

print('Config written ✓')
print(f'  unfreeze_clip_vision : True  (★ CLIP vision fine-tunes at lr=1e-5)')
print(f'  alignment_loss_coef  : {base_cfg["vera"]["alignment_loss_coef"]}   (★ was 0.10 — zero gradient, pure overhead)')
print(f'  regression_loss_coef : {base_cfg["vera"]["regression_loss_coef"]}   (★ was 0.50 — reduced so CE dominates)')
print(f'  lr (backbone)        : {base_cfg["training"]["lr"]}  (★ was 2e-4)')
print(f'  clip_vision_lr       : {base_cfg["training"]["clip_vision_lr"]}')
print(f'  batch_size           : {base_cfg["training"]["batch_size"]}  (★ was 16)')
print(f'  num_workers          : {base_cfg["training"]["num_workers"]}')
print(f'  vocab entries        : {len(base_cfg["vera"]["action_vocab"])}  (8 directions + 1 padding)')
print(f'  data_source          : {DATA_SOURCE}')


In [ ]:
# ── Cell 8: Run 6 conditions × 3 seeds ───────────────────────────────────────
#
# Conditions:
#  0  full_vera    — all 5 streams (the full model)
#  1  bc_baseline  — vision + instruction only (no E_act, no E_exp, no hist TF)
#  2  no_lang      — removes E_act + E_exp; keeps hist TF + reward gate
#  3  no_exp       — removes E_exp (Stream 3b) only; keeps E_act
#  4  no_act       — removes E_act (Stream 3a) only; keeps E_exp
#  5  no_hist_tf   — removes history sub-transformer; keeps language streams
#
# Results saved to Drive after every seed. Re-run safe (done seeds skipped).

import copy, gc, json as _json
import numpy as np, torch, yaml, psutil
from training.sft_trainer_vera import sft_train

SEEDS = [42, 123, 456]

CONDITIONS = [
    {
        'name': 'full_vera',
        'display': 'Full VERA (all 5 streams)',
        'overrides': {},
    },
    {
        'name': 'bc_baseline',
        'display': 'BC baseline (no lang, no hist TF)',
        'overrides': {
            ('vera','use_lang_feedback'):     False,
            ('vera','use_consequence_token'): False,
            ('vera','use_temporal_history'):  False,
        },
    },
    {
        'name': 'no_lang',
        'display': 'No language feedback (keeps hist TF + gate)',
        'overrides': {
            ('vera','use_lang_feedback'):     False,
            ('vera','use_consequence_token'): False,
        },
    },
    {
        'name': 'no_exp',
        'display': 'No E_exp / Stream 3b (action narration only)',
        'overrides': {
            ('vera','use_consequence_token'): False,
        },
    },
    {
        'name': 'no_act',
        'display': 'No E_act / Stream 3a (experience token only)',
        'overrides': {
            ('vera','use_lang_feedback'): False,
        },
    },
    {
        'name': 'no_hist_tf',
        'display': 'No history sub-transformer (flat positional history)',
        'overrides': {
            ('vera','use_temporal_history'): False,
        },
    },
]

with open(lt_cfg_path) as f:
    base_cfg = yaml.safe_load(f)

all_results = {}

for ci, cond in enumerate(CONDITIONS):
    cond_results = {}
    print(f'\n{"═"*62}')
    print(f'  [{ci+1}/{len(CONDITIONS)}] {cond["display"]}')
    print(f'{"═"*62}')

    for seed in SEEDS:
        run_dir  = f'{DRIVE_ROOT}/checkpoints/lt_{cond["name"]}/seed{seed}'
        log_path = f'{run_dir}/sft_vera_log.json'

        # ── Skip if already done ──────────────────────────────────────────────
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            if log:
                best = max(r['val_acc'] for r in log)
                print(f'  [SKIP] seed={seed}  best val_acc={best:.4f}')
                cond_results[f'seed{seed}'] = best
                continue

        ram_gb = psutil.virtual_memory().used / 1e9
        print(f'\n  seed={seed}  RAM={ram_gb:.1f} GB')
        os.makedirs(run_dir, exist_ok=True)

        cfg = copy.deepcopy(base_cfg)
        for (sec, key), val in cond['overrides'].items():
            cfg[sec][key] = val
        cfg['training']['seed']       = seed
        cfg['training']['output_dir'] = run_dir

        torch.manual_seed(seed)
        np.random.seed(seed)
        sft_train(cfg)

        # Free memory between runs
        torch.cuda.empty_cache()
        gc.collect()

        best = 0.0
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            best = max(r['val_acc'] for r in log) if log else 0.0
        cond_results[f'seed{seed}'] = best
        print(f'  ✓ seed={seed}  best={best:.4f}')

    all_results[cond['name']] = cond_results

summary = {'data_source': DATA_SOURCE, 'results': all_results}
_json.dump(summary, open(f'{DRIVE_ROOT}/checkpoints/lt_summary.json','w'), indent=2)
print(f'\n✓ All conditions done.')

In [ ]:
# ── Cell 9: Paper table ───────────────────────────────────────────────────────
import json, numpy as np

data = json.load(open(f'{DRIVE_ROOT}/checkpoints/lt_summary.json'))
res, src = data['results'], data.get('data_source','?')

ORDER = [
    ('bc_baseline', 'BC/SFT baseline             (no lang, no hist TF)'),
    ('no_lang',     'No lang feedback†            (hist TF + gate kept)'),
    ('no_exp',      'No E_exp / Stream 3b         (act narration only)'),
    ('no_act',      'No E_act / Stream 3a         (experience only)'),
    ('no_hist_tf',  'No history TF                (flat positional hist)'),
    ('full_vera',   'Full VERA ★                  (all 5 streams)'),
]

print(f'\nData source: {src}')
print('─'*80)
print('Language-Table ablation — validation action-classification accuracy')
print(f'{"Variant":<52} {"S42":>6} {"S123":>6} {"S456":>6} {"Mean±Std":>12}')
print('─'*80)

vera_mu = None
for key, display in ORDER:
    if key not in res:
        print(f'  {display:<52}  (not run)')
        continue
    v = [res[key].get(f'seed{s}', float('nan')) for s in [42,123,456]]
    mu, std = float(np.nanmean(v)), float(np.nanstd(v))
    if key == 'full_vera': vera_mu = mu
    vals = '  '.join(f'{x*100:5.1f}' if not np.isnan(x) else '  ---' for x in v)
    marker = ' ←best' if key == 'full_vera' else ''
    print(f'  {display:<52}  {vals}   {mu*100:.1f}±{std*100:.1f}{marker}')

print('─'*80)

# Delta over BC
if 'bc_baseline' in res and vera_mu is not None:
    bc_v = [res['bc_baseline'].get(f'seed{s}', float('nan')) for s in [42,123,456]]
    bc_mu = float(np.nanmean(bc_v))
    delta = (vera_mu - bc_mu) * 100
    sign  = '+' if delta >= 0 else ''
    print(f'  VERA vs BC delta: {sign}{delta:.2f} pp')

print()
print('† Removes E_act (Stream 3a) and E_exp (Stream 3b); hist TF and reward gate remain.')
print('\nPaste into vera_neurips.tex → tab:ablations')

In [ ]:
# ── Cell 10: Download results ─────────────────────────────────────────────────
from google.colab import files
import shutil
shutil.make_archive('/content/lt_results','zip', f'{DRIVE_ROOT}/checkpoints')
files.download('/content/lt_results.zip')
print('Download started')